<a href="https://colab.research.google.com/github/ElofssonLab/kb8029-book/blob/main/notebooks/day09-discussion-1.ipynb" style="display:inline-block;padding:10px 18px;background-color:#F9AB00;color:#000000;font-weight:bold;text-decoration:none;border-radius:6px;font-family:sans-serif;font-size:14px;">&#9654;&nbsp; Open in Google Colab</a>

# Day 9, Part 1 discussion — a different tiny backprop example

The book page and main notebook hand-computed backprop for a 2-input,
**tanh**-hidden-unit, sigmoid-output network. Here's a different tiny
network: 2 inputs, one **ReLU** hidden unit, one sigmoid output — same
shape, different activation.

**Before running anything, discuss with your group:** ReLU's derivative
is 0 for any negative input and 1 for any positive input (undefined
exactly at 0, but that never matters in practice). If the hidden unit's
pre-activation `z1` happens to be negative for this input, what do you
predict will happen to the gradient flowing back to `w1` and `w2` (the
weights that feed that hidden unit)? Write down a guess, then run the
cells below and check.

In [1]:
import torch

# Same shape as the book's worked example (2 inputs -> 1 ReLU hidden unit -> 1 sigmoid output),
# but different weights, chosen so the hidden unit's pre-activation is negative.
x1, x2 = 0.5, -0.3
w1, w2, b1 = 0.2, 0.4, -0.5   # hidden unit weights/bias
w3, b2 = 0.9, -0.1            # output unit weight/bias
y_true = 1.0

z1 = w1 * x1 + w2 * x2 + b1
a1 = max(0.0, z1)   # ReLU by hand
print(f"z1 (hidden pre-activation) = {z1:.4f}")
print(f"a1 (ReLU output)           = {a1:.4f}")
print("z1 is negative -> ReLU output is exactly 0, and ReLU's local derivative there is 0.")

z1 (hidden pre-activation) = -0.5200
a1 (ReLU output)           = 0.0000
z1 is negative -> ReLU output is exactly 0, and ReLU's local derivative there is 0.


In [2]:
import math

z2 = w3 * a1 + b2
y_hat = 1.0 / (1.0 + math.exp(-z2))
loss = -(y_true * math.log(y_hat) + (1 - y_true) * math.log(1 - y_hat))
print(f"a1={a1:.4f}  z2={z2:.4f}  y_hat={y_hat:.4f}  loss={loss:.4f}")

# Backward pass, by hand -- same chain-rule pattern as the book's worked example.
dL_dz2 = y_hat - y_true               # sigmoid + BCE loss: this simplification is standard and used in the book's example too
dL_dw3 = dL_dz2 * a1
dL_db2 = dL_dz2

dL_da1 = dL_dz2 * w3
relu_deriv = 1.0 if z1 > 0 else 0.0   # this is the key line: z1 < 0 here, so this is 0
dL_dz1 = dL_da1 * relu_deriv

dL_dw1 = dL_dz1 * x1
dL_dw2 = dL_dz1 * x2
dL_db1 = dL_dz1

print()
print("Hand-computed gradients:")
for name, val in [("dL/dw1", dL_dw1), ("dL/dw2", dL_dw2), ("dL/db1", dL_db1), ("dL/dw3", dL_dw3), ("dL/db2", dL_db2)]:
    print(f"  {name} = {val:.6f}")

a1=0.0000  z2=-0.1000  y_hat=0.4750  loss=0.7444

Hand-computed gradients:
  dL/dw1 = -0.000000
  dL/dw2 = 0.000000
  dL/db1 = -0.000000
  dL/dw3 = -0.000000
  dL/db2 = -0.524979


So: `dL/dw1`, `dL/dw2`, and `dL/db1` are all exactly **zero** — every
gradient that would update the hidden unit's own weights vanishes
completely, because ReLU's derivative is 0 wherever its input is
negative. Weight `w1`/`w2`/`b1` will not move *at all* on this training
step, no matter how large the loss is. This is the real "dying ReLU"
problem the book page mentions in the activation-function comparison,
seen directly in a concrete gradient rather than just asserted.

Now verify the hand computation against PyTorch's own `autograd`,
like the book's main worked example does:

In [3]:
x1_t = torch.tensor(0.5, dtype=torch.float64)
x2_t = torch.tensor(-0.3, dtype=torch.float64)
w1_t = torch.tensor(0.2, dtype=torch.float64, requires_grad=True)
w2_t = torch.tensor(0.4, dtype=torch.float64, requires_grad=True)
b1_t = torch.tensor(-0.5, dtype=torch.float64, requires_grad=True)
w3_t = torch.tensor(0.9, dtype=torch.float64, requires_grad=True)
b2_t = torch.tensor(-0.1, dtype=torch.float64, requires_grad=True)
y_true_t = torch.tensor(1.0, dtype=torch.float64)

z1_t = w1_t * x1_t + w2_t * x2_t + b1_t
a1_t = torch.relu(z1_t)
z2_t = w3_t * a1_t + b2_t
y_hat_t = torch.sigmoid(z2_t)
loss_t = torch.nn.functional.binary_cross_entropy(y_hat_t, y_true_t)
loss_t.backward()

hand = {"w1": dL_dw1, "w2": dL_dw2, "b1": dL_db1, "w3": dL_dw3, "b2": dL_db2}
auto = {"w1": w1_t.grad.item(), "w2": w2_t.grad.item(), "b1": b1_t.grad.item(),
        "w3": w3_t.grad.item(), "b2": b2_t.grad.item()}

print(f"{'param':6}{'hand':>14}{'autograd':>14}{'match?':>10}")
all_match = True
for name in hand:
    match = abs(hand[name] - auto[name]) < 1e-8
    all_match &= match
    print(f"{name:6}{hand[name]:>14.8f}{auto[name]:>14.8f}{str(match):>10}")
print()
print("All five gradients match to 8 decimal places." if all_match else "MISMATCH -- check the hand derivation.")

param           hand      autograd    match?
w1       -0.00000000    0.00000000      True
w2        0.00000000   -0.00000000      True
b1       -0.00000000    0.00000000      True
w3       -0.00000000   -0.00000000      True
b2       -0.52497919   -0.52497919      True

All five gradients match to 8 decimal places.


**Discuss:** the book's main network used tanh for the hidden unit,
which never has an exactly-zero derivative (its gradient just gets
*small* for large |x|, it never fully vanishes). Here, ReLU's gradient
didn't just get small — it became exactly zero. What does this suggest
about the tradeoff between ReLU (cheap, no saturation for positive
inputs, but can "die" completely for negative ones) and tanh/sigmoid
(never exactly zero, but saturates and shrinks badly across depth,
per the book's 6-layer vanishing-gradient table)? One group presents.